# TalkTalk NBA — Offline training

Trains **RandomForest** and **XGBoost** churn classifiers on the four TalkTalk tables pulled into Lovable, picks the better ROC AUC, and writes:

- `model_metrics.json` — upload via *Import results* in Lovable
- `model_artefact.pkl` — kept locally, consumed by `score_top50.ipynb`
- `feature_importance.csv` — audit trail

## Schema this notebook expects

| Table | Columns used |
|---|---|
| `customer_info` | `unique_customer_identifier`, `contract_status`, `contract_dd_cancels`, `dd_cancel_60_day`, `ooc_days`, `technology`, `speed`, `line_speed`, `sales_channel`, `crm_package_name`, `tenure_days` |
| `calls`         | `unique_customer_identifier`, `event_date`, `call_type_key`, `talk_time_seconds`, `hold_time_seconds` |
| `usage`         | `unique_customer_identifier`, `calendar_date`, `usage_download_mbs`, `usage_upload_mbs` |
| `cease`         | `unique_customer_identifier`, `cease_placed_date` (used to label churn) |

## 1 · One-time setup

```bash
pip install pandas numpy scikit-learn pyarrow fastparquet xgboost
# macOS only — XGBoost needs OpenMP at runtime:
brew install libomp
```

All four data files should sit **next to this notebook** (the default `DATA = '.'`).

In [1]:
from __future__ import annotations
import json, pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve,
)
from sklearn.model_selection import train_test_split

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed — RandomForest only')

DATA = Path('.')   # all input files are next to this notebook
OUT  = Path('.')   # outputs land here too
ID   = 'unique_customer_identifier'

## 2 · Load the four tables

Parquet is preferred; falls back to CSV automatically. `fastparquet` or `pyarrow` will satisfy the parquet engine.

In [2]:
def load(name: str) -> pd.DataFrame:
    pq = DATA / f'{name}.parquet'
    csv = DATA / f'{name}.csv'
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    raise FileNotFoundError(f'Neither {pq} nor {csv} found')

customer_info = load('customer_info')
calls         = load('calls')
usage         = load('usage')
cease         = load('cease')

for name, df in [('customer_info', customer_info), ('calls', calls), ('usage', usage), ('cease', cease)]:
    print(f'{name:14s} rows={len(df):>10,}  cols={len(df.columns)}')

customer_info  rows= 3,545,538  cols=12
calls          rows=   628,437  cols=5
usage          rows=83,185,050  cols=4
cease          rows=   146,363  cols=5


## 3 · Build the churn label

A customer is labelled `churned = 1` if they appear in `cease` with a `cease_placed_date`.

In [3]:
cease['cease_placed_date'] = pd.to_datetime(cease['cease_placed_date'], errors='coerce')
churners = set(cease.loc[cease['cease_placed_date'].notna(), ID].astype(str).unique())
print(f'Churn events: {len(churners):,}')

Churn events: 130,934


## 4 · Feature engineering

Only columns that exist in the schema are used. Engineered features:

- **`loyalty_calls_90d`** — number of `calls` rows in the last 90 days of data
- **`avg_hold_seconds`** — mean `hold_time_seconds` per customer
- **`avg_talk_seconds`** — mean `talk_time_seconds` per customer
- **`avg_download_mbs`** — mean `usage_download_mbs` per customer
- **`avg_upload_mbs`** — mean `usage_upload_mbs` per customer
- Customer-level: `ooc_days`, `tenure_days`, `speed`, `line_speed`, `contract_dd_cancels`, `dd_cancel_60_day`, plus encoded `technology`, `sales_channel`, `crm_package_name`, `contract_status`.

In [8]:
# Calls features
calls['event_date'] = pd.to_datetime(calls['event_date'], errors='coerce')
max_call = calls['event_date'].max()
recent_calls = calls[calls['event_date'] >= (max_call - pd.Timedelta(days=90))]
loyalty_90d   = recent_calls.groupby(ID).size().rename('loyalty_calls_90d')
avg_hold      = calls.groupby(ID)['hold_time_seconds'].mean().rename('avg_hold_seconds')
avg_talk      = calls.groupby(ID)['talk_time_seconds'].mean().rename('avg_talk_seconds')

# Usage features
usage['usage_download_mbs'] = pd.to_numeric(usage['usage_download_mbs'], errors='coerce')
avg_download  = usage.groupby(ID)['usage_download_mbs'].mean().rename('avg_download_mbs')
usage['usage_upload_mbs'] = pd.to_numeric(usage['usage_upload_mbs'], errors='coerce')
avg_upload    = usage.groupby(ID)['usage_upload_mbs'].mean().rename('avg_upload_mbs')

df = (customer_info
      .merge(loyalty_90d,  on=ID, how='left')
      .merge(avg_hold,     on=ID, how='left')
      .merge(avg_talk,     on=ID, how='left')
      .merge(avg_download, on=ID, how='left')
      .merge(avg_upload,   on=ID, how='left'))

df['churned'] = df[ID].astype(str).isin(churners).astype(int)
print(f'Customers: {len(df):,}  churn rate: {df["churned"].mean():.2%}')

Customers: 3,545,538  churn rate: 47.54%


In [ ]:
NUMERIC_FEATURES = [
    'ooc_days', 'tenure_days', 'speed', 'line_speed',
    'contract_dd_cancels', 'dd_cancel_60_day',
    'loyalty_calls_90d', 'avg_hold_seconds', 'avg_talk_seconds',
    'avg_download_mbs', 'avg_upload_mbs',
]
CATEGORICAL_FEATURES = ['technology', 'sales_channel', 'crm_package_name', 'contract_status']

# df is already created in cell 8 with all merges
# Just ensure it exists
NUMERIC_FEATURES = [
    'ooc_days', 'tenure_days', 'speed', 'line_speed',
    'contract_dd_cancels', 'dd_cancel_60_day',
    'loyalty_calls_90d', 'avg_hold_seconds', 'avg_talk_seconds',
    'avg_download_mbs', 'avg_upload_mbs',
]
CATEGORICAL_FEATURES = ['technology', 'sales_channel', 'crm_package_name', 'contract_status']

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print('Features used:', FEATURES)

X = df[FEATURES].copy()
for c in X.columns:
    if X[c].dtype == object:
        X[c] = X[c].astype('category').cat.codes
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['churned'].astype(int)
print('Features used:', FEATURES)

X = df[FEATURES].copy()
for c in X.columns:
    if X[c].dtype == object:
        X[c] = X[c].astype('category').cat.codes
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['churned'].astype(int)

df['churned'] = df[ID].astype(str).isin(churners).astype(int)
print('Features used:', FEATURES)

X = df[FEATURES].copy()
for c in X.columns:
    if X[c].dtype == object:
        X[c] = X[c].astype('category').cat.codes
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['churned'].astype(int)

NameError: name 'df' is not defined

## 5 · Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

## 6 · RandomForest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=20,
    n_jobs=-1, random_state=42,
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
print(f'RandomForest AUC = {rf_auc:.4f}')

## 7 · XGBoost

If you see `XGBoostError: Library not loaded: libomp.dylib`, install OpenMP via Homebrew on macOS: `brew install libomp`.

In [ ]:
best_name, best_model, best_proba, best_auc = 'RandomForest', rf, rf_proba, rf_auc

if HAS_XGB:
    xgb = XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        tree_method='hist', random_state=42,
    )
    xgb.fit(X_train, y_train)
    xgb_proba = xgb.predict_proba(X_test)[:, 1]
    xgb_auc = roc_auc_score(y_test, xgb_proba)
    print(f'XGBoost AUC = {xgb_auc:.4f}')
    if xgb_auc > best_auc:
        best_name, best_model, best_proba, best_auc = 'XGBoost', xgb, xgb_proba, xgb_auc

print(f'\n→ Selected: {best_name} (AUC={best_auc:.4f})')

## 8 · Pick decision threshold (max F1)

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
best_thresh, best_f1 = 0.5, 0.0
for t in thresholds:
    preds = (best_proba >= t).astype(int)
    f1 = f1_score(y_test, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(t)

y_pred = (best_proba >= best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred)
print(f'Threshold={best_thresh:.2f}  F1={best_f1:.4f}')
print('Confusion matrix:\n', cm)

## 9 · Per-segment metrics (by tenure bucket)

In [ ]:
seg_metrics = []
if 'tenure_days' in df.columns:
    test_idx = X_test.index
    seg_df = df.loc[test_idx, ['tenure_days']].copy()
    seg_df['pred']   = y_pred
    seg_df['actual'] = y_test.values
    bins = [(0,365,'0-12m'),(365,730,'12-24m'),(730,1460,'24-48m'),(1460,99999,'48m+')]
    for lo, hi, lab in bins:
        m = (seg_df['tenure_days'] >= lo) & (seg_df['tenure_days'] < hi)
        if m.sum() < 10:
            continue
        seg_metrics.append({
            'segment': lab,
            'precision': float(precision_score(seg_df.loc[m,'actual'], seg_df.loc[m,'pred'], zero_division=0)),
            'recall':    float(recall_score(seg_df.loc[m,'actual'], seg_df.loc[m,'pred'], zero_division=0)),
            'n':         int(m.sum()),
        })
seg_metrics

## 10 · Feature importance & ROC curve

In [ ]:
fi_arr = best_model.feature_importances_
feature_importance = sorted(
    [{'feature': f, 'importance': float(v)} for f, v in zip(FEATURES, fi_arr)],
    key=lambda d: d['importance'], reverse=True,
)

fpr, tpr, roc_t = roc_curve(y_test, best_proba)
step = max(1, len(fpr) // 50)
roc_points = [
    {'fpr': float(f), 'tpr': float(p), 'threshold': float(th)}
    for f, p, th in zip(fpr[::step], tpr[::step], roc_t[::step])
]
feature_importance[:8]

## 11 · Write outputs

Files land alongside this notebook. Upload `model_metrics.json` via *Lovable → Model → Import results*.

In [ ]:
metrics = {
    'model_type': best_name,
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'hyperparameters': best_model.get_params(),
    'performance_metrics': {
        'accuracy':           float(accuracy_score(y_test, y_pred)),
        'precision':          float(precision_score(y_test, y_pred, zero_division=0)),
        'recall':             float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_score':           float(best_f1),
        'roc_auc':            float(best_auc),
        'decision_threshold': float(best_thresh),
    },
    'confusion_matrix': {
        'true_negatives': int(cm[0,0]), 'false_positives': int(cm[0,1]),
        'false_negatives': int(cm[1,0]), 'true_positives':  int(cm[1,1]),
    },
    'dataset_split': {'train_size': int(len(X_train)), 'test_size': int(len(X_test))},
    'roc_curve': roc_points,
    'segment_metrics': seg_metrics,
    'feature_importance': feature_importance,
}

with open(OUT / 'model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

with open(OUT / 'model_artefact.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'features': FEATURES,
        'threshold': best_thresh,
        'model_type': best_name,
    }, f)

pd.DataFrame(feature_importance).to_csv(OUT / 'feature_importance.csv', index=False)
print('✓ model_metrics.json')
print('✓ model_artefact.pkl')
print('✓ feature_importance.csv')
print('\nNext: open score_top50.ipynb to produce the top-50 customers.')